# 03_1 · Dữ liệu và loại mã không phải hàng hóa
**Phần 1/4** của mô hình gợi ý sản phẩm. Các phần chạy lần lượt: 03_1 → 03_2 → 03_3 → 03_4. File `recsys_lib.py` (cùng thư mục) chứa các lớp mô hình và hàm đánh giá dùng chung.

**Mục tiêu phần này:** lấy dữ liệu tương tác từ PostgreSQL, lưu cache vào `data/`, rồi kiểm tra các mã không phải hàng hóa (túi, quà tặng, phí...) có làm baseline cao bất thường (Hit Rate@10 ≈ 0,90) hay không.

**Đầu ra:** `data/rs_*.parquet` (cache), `data/rs_state.json` (danh sách mã loại), `reports/recsys_item_nghi_van.csv`.
Cần file `.env` như notebook 01 (PGHOST, PGPORT, PGDATABASE, PGUSER, PGPASSWORD, PGSCHEMA).

In [1]:
import os
from recsys_lib import *
from IPython.display import display
URL = "postgresql+psycopg2://{u}:{p}@{h}:{port}/{db}".format(
    u=os.environ["PGUSER"], p=os.environ["PGPASSWORD"], h=os.environ["PGHOST"],
    port=os.environ.get("PGPORT", "5432"), db=os.environ["PGDATABASE"])
SCHEMA = os.environ.get("PGSCHEMA", "masked_data")
engine = create_engine(URL, connect_args={"options": f"-csearch_path={SCHEMA},public"})
def q(sql):
    with engine.connect() as con:
        return pd.read_sql(text(sql), con)
VALID = "i.is_revenue_valid::text::boolean IS TRUE AND i.is_cancelled::text::boolean IS NOT TRUE"

Latent model: SVD (scikit-learn)


## 1. Lấy dữ liệu
- `inter`: mỗi dòng là (khách, mẫu, ngày) với số lượng và doanh thu, chỉ hóa đơn hợp lệ, dòng bán, khách có mã, năm 2025.
- `item_stats`: thống kê từng mẫu trên **mọi** hóa đơn hợp lệ (kể cả khách vãng lai), dùng để phát hiện mã không phải hàng hóa.
- `catalog`: thuộc tính mẫu (ngành hàng, giới tính, mùa, bản quyền).

Kết quả được lưu cache vào `data/` để chạy lại nhanh (đặt `USE_CACHE = False` để lấy lại từ database).

In [2]:
USE_CACHE = True
cache = {n: DATA / f"rs_{n}.parquet" for n in ["inter", "item_stats", "catalog", "meta"]}
if USE_CACHE and all(p.exists() for p in cache.values()):
    inter, item_stats, catalog = (pd.read_parquet(cache[n]) for n in ["inter", "item_stats", "catalog"])
    n_inv_total = int(pd.read_parquet(cache["meta"]).n_inv_total[0])
else:
    inter = q(f"""
        SELECT NULLIF(TRIM(i.customer_id), '') AS customer_id, s.item_no AS item,
               i.invoice_date::timestamp::date AS ngay,
               SUM(d.quantity::numeric) AS qty, SUM(d.total_amount::numeric) AS rev
        FROM invoice_detail d
        JOIN invoice i ON i.invoice_id = d.invoice_id
        JOIN sku s ON UPPER(TRIM(s.sku)) = UPPER(TRIM(d.item_code))
        WHERE {VALID} AND d.quantity::numeric > 0 AND NULLIF(TRIM(i.customer_id), '') IS NOT NULL
          AND i.invoice_date::timestamp >= DATE '2025-01-01' AND i.invoice_date::timestamp < DATE '2026-01-01'
        GROUP BY 1, 2, 3""")
    item_stats = q(f"""
        SELECT s.item_no AS item,
               SUM(d.quantity::numeric) FILTER (WHERE d.quantity::numeric > 0)       AS qty,
               SUM(d.total_amount::numeric) FILTER (WHERE d.quantity::numeric > 0)   AS rev,
               COUNT(DISTINCT i.invoice_id) FILTER (WHERE d.quantity::numeric > 0)   AS n_inv,
               COUNT(*) AS n_dong,
               SUM(CASE WHEN COALESCE(d.line_type, 'item') <> 'item' THEN 1 ELSE 0 END) AS n_dong_khac_item
        FROM invoice_detail d
        JOIN invoice i ON i.invoice_id = d.invoice_id
        JOIN sku s ON UPPER(TRIM(s.sku)) = UPPER(TRIM(d.item_code))
        WHERE {VALID} GROUP BY 1""")
    n_inv_total = int(q(f"SELECT COUNT(*) AS n FROM invoice i WHERE {VALID} AND i.actual_amount::numeric > 0").n[0])
    catalog = q("SELECT item, product_name, group_cat, gender, season, licenses FROM item")
    inter["ngay"] = pd.to_datetime(inter["ngay"])
    for n, df in [("inter", inter), ("item_stats", item_stats), ("catalog", catalog)]:
        df.to_parquet(cache[n], index=False)
    pd.DataFrame({"n_inv_total": [n_inv_total]}).to_parquet(cache["meta"], index=False)

inter["item"] = inter["item"].astype(str); item_stats["item"] = item_stats["item"].astype(str); catalog["item"] = catalog["item"].astype(str)
inter["ngay"] = pd.to_datetime(inter["ngay"])
for c in ["qty", "rev"]: inter[c] = inter[c].astype(float)
print(f"{len(inter):,} dòng (khách, mẫu, ngày) | {inter.customer_id.nunique():,} khách | {inter.item.nunique():,} mẫu | "
      f"{inter.ngay.min():%d/%m/%Y} – {inter.ngay.max():%d/%m/%Y}")

350,663 dòng (khách, mẫu, ngày) | 67,307 khách | 2,832 mẫu | 01/01/2025 – 30/12/2025


## 2. Kiểm tra mã không phải hàng hóa
Baseline ở notebook 01 có Hit Rate@10 ≈ 0,90, cao bất thường với hơn 2.800 mẫu. Nguyên nhân có thể là các mã đi kèm hầu hết hóa đơn
(túi, quà tặng, phí) nằm trong top bán chạy. Bảng top-50 theo doanh thu không phát hiện được vì các mã này có doanh thu bằng 0.

Quy tắc **tự động loại**: tổng doanh thu ≤ 0, hoặc giá bán trung bình < 5.000đ, hoặc hơn nửa số dòng có `line_type` khác `item`.
Mẫu xuất hiện trong hơn 10% hóa đơn chỉ được **đánh dấu để bạn xem lại** (có thể là sản phẩm thật bán rất chạy).

In [3]:
s = item_stats.merge(catalog[["item", "product_name", "group_cat"]], on="item", how="left")
s["gia_tb"] = s.rev / s.qty.replace(0, np.nan)
s["ty_le_hd_pct"] = 100 * s.n_inv / n_inv_total
s["pct_dong_khac_item"] = s.n_dong_khac_item / s.n_dong
s["tu_dong_loai"] = (s.rev.fillna(0) <= 0) | (s.gia_tb < 5000) | (s.pct_dong_khac_item > 0.5)
s["nghi_van"] = s.ty_le_hd_pct > 10

print("Top 20 mẫu theo SỐ LƯỢNG bán (kiểm tra có phải hàng thật không):")
display(s.sort_values("qty", ascending=False).head(20)[["item", "product_name", "group_cat", "qty", "rev", "gia_tb", "ty_le_hd_pct", "tu_dong_loai", "nghi_van"]])
print("Top 20 mẫu theo TỶ LỆ HÓA ĐƠN chứa mẫu đó:")
display(s.sort_values("ty_le_hd_pct", ascending=False).head(20)[["item", "product_name", "group_cat", "qty", "rev", "gia_tb", "ty_le_hd_pct", "tu_dong_loai"]])

Top 20 mẫu theo SỐ LƯỢNG bán (kiểm tra có phải hàng thật không):


,item,product_name,group_cat,qty,rev,gia_tb,ty_le_hd_pct,tu_dong_loai,nghi_van
169,20001000,Túi đựng hàng Rabity loại nhỏ,06. Others,"66,611.0000",0.0000,0.0000,25.0205,True,True
171,20002000,Túi đựng hàng Rabity loại vừa,06. Others,"33,195.0000",1.0000,0.0000,12.3558,True,True
3223,99200125,992.001 Túi giấy Rabity nhỏ,06. Others,"30,374.0000",0.0000,0.0000,11.5892,True,True
405,33036,Set 2 quần lót BG 33.036,06. Others,"18,696.0000","2,485,844,742.0000","132,961.3148",4.4340,False,False
3224,99200225,Túi giấy Rabity trung - 992.002,06. Others,"14,896.0000",0.0000,0.0000,5.6621,True,False
1341,70032000,Quà tặng kẹo túi Cham quà bí mật - 70.032,06. Others,"6,791.0000","1,402,470.0000",206.5189,1.3135,True,False
179,20003000,Túi đựng hàng Rabity loại to,06. Others,"4,223.0000",0.0000,0.0000,1.5548,True,False
410,33038,Set 2 quần lót bé gái - 33.038,06. Others,"3,753.0000","505,214,552.0000","134,616.1876",0.9905,False,False
2867,95200624,Đầm thô SN BG 952.006,03. Dress (đầm & yếm),"3,289.0000","865,895,507.0000","263,270.1450",1.2911,False,False
243,22076000,[QUÀ TẶNG] Set 5 bao lì xì Tài Lộc Rabity 22076,06. Others,"2,865.0000","59,504.0000",20.7693,1.1659,True,False


Top 20 mẫu theo TỶ LỆ HÓA ĐƠN chứa mẫu đó:


,item,product_name,group_cat,qty,rev,gia_tb,ty_le_hd_pct,tu_dong_loai
169,20001000,Túi đựng hàng Rabity loại nhỏ,06. Others,"66,611.0000",0.0000,0.0000,25.0205,True
171,20002000,Túi đựng hàng Rabity loại vừa,06. Others,"33,195.0000",1.0000,0.0000,12.3558,True
3223,99200125,992.001 Túi giấy Rabity nhỏ,06. Others,"30,374.0000",0.0000,0.0000,11.5892,True
3224,99200225,Túi giấy Rabity trung - 992.002,06. Others,"14,896.0000",0.0000,0.0000,5.6621,True
405,33036,Set 2 quần lót BG 33.036,06. Others,"18,696.0000","2,485,844,742.0000","132,961.3148",4.4340,False
179,20003000,Túi đựng hàng Rabity loại to,06. Others,"4,223.0000",0.0000,0.0000,1.5548,True
1341,70032000,Quà tặng kẹo túi Cham quà bí mật - 70.032,06. Others,"6,791.0000","1,402,470.0000",206.5189,1.3135,True
2867,95200624,Đầm thô SN BG 952.006,03. Dress (đầm & yếm),"3,289.0000","865,895,507.0000","263,270.1450",1.2911,False
243,22076000,[QUÀ TẶNG] Set 5 bao lì xì Tài Lộc Rabity 22076,06. Others,"2,865.0000","59,504.0000",20.7693,1.1659,True
410,33038,Set 2 quần lót bé gái - 33.038,06. Others,"3,753.0000","505,214,552.0000","134,616.1876",0.9905,False


In [4]:
# Sau khi xem hai bảng trên, thêm mã cần loại thủ công vào đây (ví dụ túi, quà tặng chưa bị quy tắc bắt được):
NON_MERCH_MANUAL = {"71110000"}          # ví dụ: {"TUI001", "QUA002"}

blacklist = set(s.loc[s.tu_dong_loai, "item"]) | NON_MERCH_MANUAL
share_removed = inter.item.isin(blacklist).mean()
print(f"Loại {len(blacklist)} mã không phải hàng hóa, chiếm {100*share_removed:.2f}% số dòng tương tác")
print("Mẫu nghi vấn (>10% hóa đơn) giữ lại:", s.loc[s.nghi_van & ~s.item.isin(blacklist), "item"].tolist())

# Lưu kết quả cho các phần sau
save_state(blacklist=sorted(blacklist))
s.sort_values("qty", ascending=False).head(200).to_csv(REPORTS / "recsys_item_nghi_van.csv", index=False)
print("Đã lưu danh sách", len(blacklist), "mã loại vào", STATE)

Loại 35 mã không phải hàng hóa, chiếm 25.44% số dòng tương tác
Mẫu nghi vấn (>10% hóa đơn) giữ lại: []
Đã lưu danh sách 35 mã loại vào data\rs_state.json
